In [34]:
from pathlib import Path
import sys

# Find the project root from the notebook's working directory.
PROJECT_ROOT = Path.cwd().resolve()

# Run until found the (PROJECT_ROOT / "requirements.txt")
while not (PROJECT_ROOT / "requirements.txt").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Cannot find the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

# Allow imports starting with "src".
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

In [53]:
import numpy as np

train_data = np.load(DATA_DIR / "mnist_train.npz")
test_data = np.load(DATA_DIR / "mnist_test.npz")

X_train, y_train = train_data["X"], train_data["y"]
X_test, y_test = test_data["X"], test_data["y"]

In [48]:
from src.preprocessing.scaler import scale_flat, prepare, scale_cnn, split_val
from src.models.logistic_regression import LogisticRegression
from src.evaluation.metrics import ClassificationMetrics
from src.models.ensemble_method_mlp import MLPEnsemble
from src.models.cnn import CNN
# from src.models.naive_bayes import GaussianNaiveBayes

In [49]:
X_train, X_val, y_train, y_val = split_val(
    X=X_train,y=y_train,
    val_size=0.1, seed=42
)

In [50]:
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'X_val shape: {X_test.shape}')

X_train shape: (48600, 784)
X_test shape: (10000, 784)
X_val shape: (10000, 784)


In [54]:
X_train_logis, X_val_logis, X_test_logis, tools = scale_flat(X_train=X_train, X_test=X_test, X_val=X_val, mode='standard')

In [55]:
model = LogisticRegression(
    eta=0.1,
    lam= 0.001,
    max_epochs=100,
    batch_size=128,
    seed=42,
    tol=1e-7
    )

In [58]:
model.fit(X_train_logis, y_train)

In [60]:
y_pred = model.predict(X_test_logis)

In [61]:
metrics = ClassificationMetrics(y_true=y_test, y_pred=y_pred)

In [62]:
metrics.report()

     Class  Precision     Recall         F1    Support
         0      0.958      0.971      0.965 
         1      0.958      0.981      0.969 
         2      0.927      0.891      0.909 
         3      0.891      0.927      0.908 
         4      0.929      0.933      0.931 
         5      0.912      0.861      0.886 
         6      0.941      0.950      0.945 
         7      0.937      0.917      0.927 
         8      0.874      0.885      0.880 
         9      0.909      0.915      0.912 
 Macro avg      0.924      0.923      0.923 

Accuracy: 0.924


In [28]:
config = {
    "sizes": (784, 128, 10),
    "activation": "relu",
    "init": "he",
    "p_drop": 0.4,
    "seeds": (42, 123, 2026),

    "optimizer": "sgd",
    "momentum": 0.9,
    "lr": 0.05,
    "batch_size": 128,
    "max_epochs": 30,
    "patience": 5,
    "weight_decay": 1e-4,

    "input_kind": "standard",
    "val_size": 0.1,
}

In [40]:
model_ensemble_mlp = MLPEnsemble(
    sizes=config['sizes'],
    seeds=config['seeds'],
    p_drop=config['p_drop'],
    init=config['init'],
)

In [41]:
model_ensemble_mlp.fit(
    X=X_train,
    y=y_train,
    validation_data=(X_val,y_val),
    weight_decay=1e-4,
    patience=5,
    momentum=config["momentum"],
)


Training member 1/3 (seed=42)
Epoch 01/30 | train loss 0.1729 acc 0.9474 | val loss 0.1939 acc 0.9422
Epoch 02/30 | train loss 0.1271 acc 0.9626 | val loss 0.1495 acc 0.9560
Epoch 03/30 | train loss 0.1022 acc 0.9709 | val loss 0.1286 acc 0.9630
Epoch 04/30 | train loss 0.0840 acc 0.9758 | val loss 0.1155 acc 0.9662
Epoch 05/30 | train loss 0.0783 acc 0.9769 | val loss 0.1093 acc 0.9680
Epoch 06/30 | train loss 0.0710 acc 0.9790 | val loss 0.1065 acc 0.9680
Epoch 07/30 | train loss 0.0613 acc 0.9818 | val loss 0.0994 acc 0.9712
Epoch 08/30 | train loss 0.0578 acc 0.9826 | val loss 0.0960 acc 0.9728
Epoch 09/30 | train loss 0.0555 acc 0.9830 | val loss 0.0965 acc 0.9732
Epoch 10/30 | train loss 0.0519 acc 0.9844 | val loss 0.0956 acc 0.9712
Epoch 11/30 | train loss 0.0494 acc 0.9854 | val loss 0.0906 acc 0.9753
Epoch 12/30 | train loss 0.0449 acc 0.9868 | val loss 0.0873 acc 0.9780
Epoch 13/30 | train loss 0.0433 acc 0.9875 | val loss 0.0872 acc 0.9760
Epoch 14/30 | train loss 0.0415 a

In [42]:
y_pred = model_ensemble_mlp.predict(X=X_test)

In [43]:
metrics_ensemble = ClassificationMetrics(y_true=y_test, y_pred=y_pred)

In [44]:
metrics_ensemble.report()

     Class  Precision     Recall         F1    Support
         0      0.979      0.991      0.985 
         1      0.986      0.989      0.988 
         2      0.982      0.983      0.982 
         3      0.980      0.985      0.983 
         4      0.981      0.978      0.979 
         5      0.983      0.978      0.980 
         6      0.983      0.981      0.982 
         7      0.975      0.978      0.976 
         8      0.980      0.974      0.977 
         9      0.981      0.972      0.977 
 Macro avg      0.981      0.981      0.981 

Accuracy: 0.981


In [ ]:
from src.models.mlp import MLP

In [ ]:
model_mlp = MLP(
    sizes=config['sizes'],
    activation='relu',
    p_drop=0.3,
    init='he',
    seed=42
    )

In [ ]:
model_mlp.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=128,
    lr=0.05,
)

Epoch 01/20 | train loss 0.1655 acc 0.9505
Epoch 02/20 | train loss 0.1151 acc 0.9664
Epoch 03/20 | train loss 0.0906 acc 0.9739
Epoch 04/20 | train loss 0.0731 acc 0.9787
Epoch 05/20 | train loss 0.0670 acc 0.9803
Epoch 06/20 | train loss 0.0566 acc 0.9830
Epoch 07/20 | train loss 0.0494 acc 0.9853
Epoch 08/20 | train loss 0.0443 acc 0.9871
Epoch 09/20 | train loss 0.0420 acc 0.9879
Epoch 10/20 | train loss 0.0354 acc 0.9899
Epoch 11/20 | train loss 0.0344 acc 0.9898
Epoch 12/20 | train loss 0.0305 acc 0.9917
Epoch 13/20 | train loss 0.0282 acc 0.9924
Epoch 14/20 | train loss 0.0274 acc 0.9925
Epoch 15/20 | train loss 0.0257 acc 0.9926
Epoch 16/20 | train loss 0.0241 acc 0.9934
Epoch 17/20 | train loss 0.0216 acc 0.9938
Epoch 18/20 | train loss 0.0200 acc 0.9946
Epoch 19/20 | train loss 0.0198 acc 0.9945
Epoch 20/20 | train loss 0.0167 acc 0.9960


{'train_loss': [0.16553913525961064,
  0.11510531935426924,
  0.09060074016451836,
  0.0731476532916228,
  0.0669972206707354,
  0.05662860728248402,
  0.04936602463324865,
  0.044330567535426885,
  0.04195091383600676,
  0.035374067033882496,
  0.03436463115805829,
  0.03046567048187609,
  0.028151325778000884,
  0.02742408250493032,
  0.0257134980832537,
  0.024060554181536038,
  0.021550161384597973,
  0.01997755467891693,
  0.019828740679831418,
  0.01665902956975279],
 'train_acc': [0.9504814814814815,
  0.9664074074074074,
  0.973925925925926,
  0.9786851851851852,
  0.9802592592592593,
  0.983,
  0.9852592592592593,
  0.9870925925925926,
  0.9878888888888889,
  0.9899074074074075,
  0.9898333333333333,
  0.9917407407407407,
  0.9923888888888889,
  0.9924814814814815,
  0.9925925925925926,
  0.9933888888888889,
  0.9938333333333333,
  0.9946296296296296,
  0.994537037037037,
  0.9959814814814815],
 'val_loss': [],
 'val_acc': []}

In [ ]:
y_pred = model_mlp.predict(X_test)

In [ ]:
metrics_mlp = ClassificationMetrics(y_true=y_test, y_pred=y_pred)

In [ ]:
metrics_mlp.report()

     Class  Precision     Recall         F1    Support
         0      0.980      0.993      0.986 
         1      0.991      0.989      0.990 
         2      0.981      0.975      0.978 
         3      0.978      0.983      0.981 
         4      0.980      0.977      0.978 
         5      0.982      0.973      0.977 
         6      0.985      0.973      0.979 
         7      0.979      0.979      0.979 
         8      0.963      0.985      0.974 
         9      0.979      0.970      0.975 
 Macro avg      0.980      0.980      0.980 

Accuracy: 0.980
